# CNN vs ViT explanation faithfulness - full pipeline
Runtime -> Change runtime type -> **GPU (A100/L4 if you have Colab Pro)**, then run top to bottom. Finished steps are skipped, so after a disconnect just rerun the setup cell and Run all.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/xai-cnn-vit'
!mkdir -p $CKPT_DIR

import os, sys
REPO = 'https://github.com/hsfcoulibaly/Explanability-Methods.git'   # repo ROOT, not /tree/...
if not os.path.exists('/content/repo'):
    !git clone -q $REPO /content/repo
else:
    !cd /content/repo && git pull -q
PROJECT = '/content/repo/ViT'   # local Colab folder that contains src/
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
!pip install -q timm captum
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - change runtime!')

## 1. Fine-tune (3 epochs is enough with pretrained weights; raise EPOCHS for the 10/20/30 study)

In [ ]:
EPOCHS = 3
for arch in ['vit', 'cnn']:
    ck = f'{CKPT_DIR}/{arch}.pt'
    if os.path.exists(ck): print(arch, 'already trained, skipping'); continue
    !python -m src.train --arch {arch} --epochs {EPOCHS} --batch-size 128 --out {ck}

## 2. Quick check (a few batches per pair)

In [ ]:
for arch in ['vit', 'cnn']:
    !python -m src.run_eval --arch {arch} --ckpt $CKPT_DIR/{arch}.pt --out $CKPT_DIR/quick.json --max-batches 2

## 3. Full run: all 1,000 images, blur removal (writes results.json to Drive)

In [ ]:
import json
done = json.load(open(f'{CKPT_DIR}/results.json')) if os.path.exists(f'{CKPT_DIR}/results.json') else {}
for arch, methods in [('cnn', ['gradcam','ig','random']), ('vit', ['gradcam','ig','rollout','random'])]:
    todo = [m for m in methods if f'{arch}/{m}/blur' not in done]
    if todo:
        !python -m src.run_eval --arch {arch} --ckpt $CKPT_DIR/{arch}.pt --out $CKPT_DIR/results.json --methods {' '.join(todo)}

## 4. Robustness check: mean-color removal

In [ ]:
done = json.load(open(f'{CKPT_DIR}/results.json'))
for arch, methods in [('cnn', ['gradcam','ig','random']), ('vit', ['gradcam','ig','rollout','random'])]:
    todo = [m for m in methods if f'{arch}/{m}/mean' not in done]
    if todo:
        !python -m src.run_eval --arch {arch} --ckpt $CKPT_DIR/{arch}.pt --out $CKPT_DIR/results.json --removal mean --methods {' '.join(todo)}

## 5. Results table + heatmaps

In [ ]:
import pandas as pd
res = json.load(open(f'{CKPT_DIR}/results.json'))
rows = [dict(zip(['model','method','removal'], k.split('/')),
             deletion=v['deletion_auc'], insertion=v['insertion_auc'], n=v['n'], test_acc=v.get('test_acc'))
        for k, v in res.items()]
df = pd.DataFrame(rows).round(4); df.to_csv(f'{CKPT_DIR}/faithfulness.csv', index=False)
df

In [ ]:
import matplotlib.pyplot as plt
from src.models import load_checkpoint
from src.data import get_eval_subset, denormalize
from src.explain import get_explainer
device = 'cuda'
x, y = next(iter(get_eval_subset(batch_size=6))); x = x.to(device)
fig, ax = plt.subplots(8, 6, figsize=(14, 19))
r = 0
for arch, methods in [('cnn', ['gradcam','ig']), ('vit', ['gradcam','ig','rollout'])]:
    model, _ = load_checkpoint(arch, f'{CKPT_DIR}/{arch}.pt', device)
    for m in methods:
        s, _ = get_explainer(m)(model, x)
        for i in range(6):
            ax[r+1, i].imshow(denormalize(x[i]).permute(1,2,0).cpu()); ax[r+1, i].imshow(s[i].cpu(), cmap='jet', alpha=.5)
        ax[r+1, 0].set_ylabel(f'{arch} {m}'); r += 1
for i in range(6): ax[0, i].imshow(denormalize(x[i]).permute(1,2,0).cpu())
for a in ax.flat: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(f'{CKPT_DIR}/heatmaps.png', dpi=130); plt.show()

## 6. Optional: randomization sanity check

In [ ]:
for arch in ['vit', 'cnn']:
    !python -m src.sanity --arch {arch} --ckpt $CKPT_DIR/{arch}.pt --out $CKPT_DIR/sanity_{arch}.json --n-images 64